In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import jax.numpy as jnp
import jax
import jax.random as random
from flax import linen as nn
import optax
from tqdm import tqdm
from utils import DataLoader
from utils import MLP

In [4]:
key = random.PRNGKey(0) # chiave per random 

f_to_learn = lambda mu, k, l, x: jnp.sin(2*mu*jnp.pi*x) + k + jnp.exp(l*x)
N = 10000

key, subkey = random.split(key) # ogni volta, prima di usare la chiave, la devi dividere
x = random.uniform(subkey, (N,), minval=-10, maxval=10)
key, subkey = random.split(key)
mu = random.uniform(subkey, (N,), minval=-2, maxval=2)
key, subkey = random.split(key)
k = random.uniform(subkey, (N,), minval=-5, maxval=5)
key, subkey = random.split(key)
l = random.uniform(subkey, (N,), minval=-1, maxval=1)

y = f_to_learn(mu, k, l, x) # così generiamo artificialmente un dataset di N punti

In [5]:
X = jnp.stack([mu, k, l, x], axis=1)

In [6]:
X = jnp.stack([mu, k, l, x], axis=1)


split_idx = int(N * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

train_dataloader = DataLoader(X_train, y_train, batch_size=32, shuffle=True)
test_dataloader = DataLoader(X_test, y_test, batch_size=32, shuffle=False)


In [7]:
# Example of iterating through the DataLoader
for data, label in train_dataloader:
    print(data.shape, label.shape)
    break

(32, 4) (32, 1)


In [8]:
targetnetwork = MLP(output_dim=1, hidden_dim=8, num_hidden_layers=1)

In [9]:
x = jnp.ones((1,1)) #Gli input sono SEMPRE (SEMPRE) nel formato (bathc_size, input_dim1, input_dim2, ..., input_dimN)
# In questo caso, batch_size=1, input_dim=1
key = jax.random.PRNGKey(0) # Bisogna sempre passare una key per inizializzare i pesi random
print(targetnetwork.tabulate(key, x)) # Visualizza la struttura del modello, con i pesi inizializzati


                               MLP Summary                               
┏━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ path    ┃ module ┃ inputs       ┃ outputs      ┃ params               ┃
┡━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│         │ MLP    │ float32[1,1] │ float32[1,1] │                      │
├─────────┼────────┼──────────────┼──────────────┼──────────────────────┤
│ Dense_0 │ Dense  │ float32[1,1] │ float32[1,8] │ bias: float32[8]     │
│         │        │              │              │ kernel: float32[1,8] │
│         │        │              │              │                      │
│         │        │              │              │ 16 (64 B)            │
├─────────┼────────┼──────────────┼──────────────┼──────────────────────┤
│ Dense_1 │ Dense  │ float32[1,8] │ float32[1,1] │ bias: float32[1]     │
│         │        │              │              │ kernel: float32[8,1] │
│         │        │              │  

In [10]:
hypernetwork = MLP(output_dim = 25, hidden_dim=8, num_hidden_layers=2) # 25 come i parametri del target network

## First try: only training a single network

In [11]:
key = jax.random.PRNGKey(0)
key, subkey = random.split(key)
toy_data = random.uniform(key, (1000, 1), minval=-3, maxval=3)
toy_label = f_to_learn(0.5, 1.0, 0.1, toy_data)
train_dataloader = DataLoader(toy_data, toy_label, batch_size=32, shuffle=True)

In [12]:
model = MLP(output_dim=1, hidden_dim=8, num_hidden_layers=1)

In [13]:
def mse_loss(preds, targets):
    return jnp.mean((preds - targets) ** 2)

optimizer = optax.adam(learning_rate=1e-3)
params = model.init(jax.random.PRNGKey(0), jnp.zeros((1, 1)))
opt_state = optimizer.init(params)
epochs = 1000

In [14]:
# HO CAPITO GLI ITERATORI LOL
it = iter(train_dataloader)
next(it)

(Array([[ 1.3251486 ],
        [ 1.7259712 ],
        [-1.4758916 ],
        [ 1.886148  ],
        [ 1.1210532 ],
        [-0.7078543 ],
        [ 2.0538845 ],
        [ 0.8367226 ],
        [-2.7543204 ],
        [-2.8492084 ],
        [-1.7900848 ],
        [-1.6452813 ],
        [ 0.29829955],
        [ 2.9617748 ],
        [ 1.2408056 ],
        [-0.33282137],
        [-0.01171947],
        [ 1.8245993 ],
        [-2.411188  ],
        [-2.432755  ],
        [ 0.43191075],
        [ 0.0389936 ],
        [-1.9057281 ],
        [ 2.5203166 ],
        [ 0.5336602 ],
        [ 1.9736738 ],
        [ 0.84615755],
        [-1.7945638 ],
        [ 2.6997862 ],
        [-1.15661   ],
        [-1.8660572 ],
        [ 2.6787367 ]], dtype=float32),
 Array([[1.2888119 ],
        [1.4299663 ],
        [2.8599188 ],
        [1.8574771 ],
        [1.7474318 ],
        [1.1373932 ],
        [2.3964782 ],
        [2.5780232 ],
        [1.0617999 ],
        [1.2958694 ],
        [2.4487953 ],
     

In [15]:
def train_step(model, params, opt_state, loss, optimizer, x, y, key=None):
    loss_fn = lambda p, x, y: loss(model.apply(p, x), y)
    loss, grad = jax.value_and_grad(loss_fn)(params, x, y)
    
    updates, opt_state = optimizer.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)

    return params, opt_state, loss

train_step = jax.jit(train_step, static_argnames=('model', 'loss', 'optimizer'))

for epoch in tqdm(range(epochs)):
    epoch_loss = 0.0
    for data, label in train_dataloader:
        params, opt_state, loss = train_step(model = model, params = params, opt_state = opt_state, loss = mse_loss, optimizer = optimizer, x = data, y = label)
        epoch_loss += loss
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {epoch_loss / len(train_dataloader)}")

print(f"Final loss: {epoch_loss / len(train_dataloader)}")

# TRAINA, POI VEDIAMO SE OVERFITTA CON UN TEST SET

  0%|          | 2/1000 [00:01<11:08,  1.49it/s]

Epoch 0, Loss: 3.7409934997558594


  1%|          | 12/1000 [00:02<02:13,  7.39it/s]

Epoch 10, Loss: 1.0597807168960571


  2%|▏         | 22/1000 [00:04<01:54,  8.54it/s]

...

Epoch 980, Loss: 0.10234957188367844


 99%|█████████▉| 992/1000 [02:01<00:00,  8.65it/s]

Epoch 990, Loss: 0.10213912278413773


100%|██████████| 1000/1000 [02:02<00:00,  8.15it/s]

Final loss: 0.10195412486791611


In [16]:
labels = random.uniform(key, (10, 1), minval=-3, maxval=3)
labels = labels.squeeze()
labels

a = labels[3:6]
a

Array([-2.2756462, -1.8491192,  1.3320904], dtype=float32)

In [17]:
jnp.expand_dims(a, axis=-1)

Array([[-2.2756462],
       [-1.8491192],
       [ 1.3320904]], dtype=float32)

In [18]:
labels = random.uniform(key, (10, 3), minval=-3, maxval=3)
labels = labels.squeeze()
labels

a = labels[1:6, :]
a

Array([[-2.2756462 , -1.8491192 ,  1.3320904 ],
       [ 1.5926733 , -2.0847573 ,  2.7102375 ],
       [-2.8241372 , -2.4077368 ,  0.31885958],
       [-2.2533174 ,  0.5673723 ,  2.7569447 ],
       [ 1.1593628 ,  1.3445759 , -1.0910139 ]], dtype=float32)

In [19]:
a.shape

(5, 3)

In [20]:
jnp.expand_dims(a, axis=-1)

Array([[[-2.2756462 ],
        [-1.8491192 ],
        [ 1.3320904 ]],

       [[ 1.5926733 ],
        [-2.0847573 ],
        [ 2.7102375 ]],

       [[-2.8241372 ],
        [-2.4077368 ],
        [ 0.31885958]],

       [[-2.2533174 ],
        [ 0.5673723 ],
        [ 2.7569447 ]],

       [[ 1.1593628 ],
        [ 1.3445759 ],
        [-1.0910139 ]]], dtype=float32)

https://huggingface.co/blog/afmck/flax-tutorial
https://wandb.ai/jax-series/simple-training-loop/reports/Writing-a-Training-Loop-in-JAX-and-Flax--VmlldzoyMzA4ODEy